# 10 · Not marrying an SDK: swapping circuit emitters

Quantum SDKs churn. Qiskit's 1.0 transition removed `opflow`, moved `qiskit.algorithms`
out of the core package, deleted `execute()`, and revised its primitives twice — plenty
of published quantum-ML code simply does not run anymore.

`qbmkit` is built so that **this cannot break the library**:

> The QBM algorithms live in a small **internal circuit IR**, executed by our own
> dependency-free simulator. Qiskit, PennyLane and OpenQASM are *emitters* — thin leaf
> adapters. If an SDK makes a breaking change, you fix **one file**, not the library.

This notebook shows the claim is real, and how to switch engines.

## Step 1 — the layering, in numbers

Let's not take it on faith. Count the SDK imports in the source tree and the size of
each layer.

In [1]:
import subprocess, pathlib, textwrap
import numpy as np
import qbm

root = pathlib.Path(qbm.__file__).parent

def loc(p):
    return sum(1 for _ in open(p))

core = sorted((root / "circuits").glob("*.py"))
adapters = sorted((root / "circuits" / "adapters").glob("*.py"))

print("CORE circuit layer (algorithms live here):")
for f in core:
    print(f"   {f.name:18s} {loc(f):4d} lines")
print(f"   {'TOTAL':18s} {sum(loc(f) for f in core):4d} lines")
print("\nADAPTERS (vendor emitters -- the only churn surface):")
for f in adapters:
    print(f"   {f.name:24s} {loc(f):4d} lines")
print(f"   {'TOTAL':24s} {sum(loc(f) for f in adapters):4d} lines")

# where do the SDK names actually appear?
hits = subprocess.run(
    ["grep", "-rn", "-E", r"(^|[^a-z_])(import (qiskit|pennylane)|from (qiskit|pennylane))",
     str(root), "--include=*.py"], capture_output=True, text=True).stdout.strip().splitlines()
print("\nEvery qiskit/pennylane import in the whole package:")
for h in hits:
    print("   ", h.replace(str(root), "qbm").strip())

CORE circuit layer (algorithms live here):
   __init__.py          37 lines
   builder.py          150 lines
   densities.py        110 lines
   estimators.py       127 lines
   ir.py               130 lines
   simulator.py        108 lines
   TOTAL               662 lines

ADAPTERS (vendor emitters -- the only churn surface):
   __init__.py                69 lines
   pennylane_adapter.py       72 lines
   qasm.py                    98 lines
   qiskit_adapter.py          89 lines
   TOTAL                     328 lines

Every qiskit/pennylane import in the whole package:
    qbm/circuits/adapters/qiskit_adapter.py:18:    from qiskit import QuantumCircuit
    qbm/circuits/adapters/qiskit_adapter.py:19:    from qiskit.circuit.library import UnitaryGate
    qbm/circuits/adapters/qiskit_adapter.py:68:    from qiskit.quantum_info import Statevector
    qbm/circuits/adapters/qiskit_adapter.py:83:    from qiskit import transpile
    qbm/circuits/adapters/pennylane_adapter.py:16:    import penn

Every single SDK reference is inside an adapter — and inside a *function*, so even
importing the adapter module does not require the SDK.

## Step 2 — prove it: run with every SDK blocked

The strongest test is to make the SDKs un-importable and check the circuit pipeline
still works end to end. We install an import blocker, then compute an expectation value
and an α-z information matrix through circuits.

In [2]:
import sys, importlib.abc

BLOCKED = {"qiskit", "pennylane", "jax", "jaxlib", "quimb"}

class Blocker(importlib.abc.MetaPathFinder):
    def find_spec(self, fullname, path=None, target=None):
        if fullname.split(".")[0] in BLOCKED:
            raise ImportError(f"BLOCKED: {fullname}")
        return None

blocker = Blocker()
sys.meta_path.insert(0, blocker)
try:
    import qiskit
    print("blocker failed!")
except ImportError as e:
    print("qiskit is now un-importable:", e)

from qbm.backends.circuit import CircuitBackend
from qbm.metrics import AlphaZ

ham   = qbm.ParamHamiltonian(qbm.local_pauli_generators(3))
theta = np.random.default_rng(0).normal(scale=0.4, size=ham.n_params)
dense = qbm.DenseBackend().thermal_state(ham, theta)
O     = qbm.hamiltonians.tfim(3, g=1.2)

st = CircuitBackend(seed=0).thermal_state(ham, theta)
print("\nfull circuit pipeline with NO quantum SDK available:")
print(f"   <O> vs dense           : {abs(st.expect(O) - dense.expect(O)):.1e}")
m = st.metric(AlphaZ(0.5, 1.0))
ref = dense.metric("wigner_yanase")
print(f"   alpha-z metric vs dense: {np.max(np.abs(m - ref)) / np.max(np.abs(ref)):.3f} relative")
print(f"   backends usable        : {qbm.available_backends()}")
print(f"   adapters usable        : {qbm.circuits.adapters.available_adapters()}")

sys.meta_path.remove(blocker)   # restore for the rest of the notebook
print("\n(blocker removed)")

qiskit is now un-importable: BLOCKED: qiskit

full circuit pipeline with NO quantum SDK available:
   <O> vs dense           : 1.1e-16
   alpha-z metric vs dense: 0.033 relative
   backends usable        : ['circuit', 'dense', 'statevector']
   adapters usable        : ['qasm3']

(blocker removed)


Nothing was missing. Notice `available_backends()` and `available_adapters()` also
*correctly shrank* to what the environment can actually support — the library reports
its real capabilities rather than failing later.

(Importing `qbm` does not import any SDK, so blocking them afterwards genuinely
removes them: optional backends are registered as lazy factories and only load their
dependency when you construct one.)

## Step 3 — the same circuit, three ways

Now the practical part. A circuit is just an IR object; you choose what runs it.

In [3]:
from qbm.circuits import Circuit, builder, simulator

c = Circuit(3, name="demo")
c.h(0); c.cx(0, 1); c.ry(0.4, 2); c.cz(1, 2); c.rz(0.7, 0)
print(c)
print("gate counts:", c.gate_counts(), "| depth:", c.depth)
print("\ninstructions:")
for g in c.gates:
    print("   ", g)

Circuit('demo', n_qubits=3, gates=5, depth=3)
gate counts: {'cx': 1, 'cz': 1, 'h': 1, 'ry': 1, 'rz': 1} | depth: 3

instructions:
    h [0]
    cx [0, 1]
    ry(0.4) [2]
    cz [1, 2]
    rz(0.7) [0]


### (a) OpenQASM 3 — no dependency, maximum durability

QASM is the interchange standard: every SDK and most hardware providers import it, and
it outlives vendor refactors. This is the format to archive alongside a paper.

In [4]:
from qbm.circuits.adapters import to_qasm3
print(to_qasm3(c))

OPENQASM 3.0;
include "stdgates.inc";
qubit[3] q;
bit[3] c;
h q[0];
cx q[0], q[1];
ry(0.4) q[2];
cz q[1], q[2];
rz(0.7) q[0];
c = measure q;


### (b) and (c) Qiskit and PennyLane

Same IR object, different emitter. Each adapter is a pure translation of gates.

In [5]:
from qbm.circuits.adapters import to_qiskit, to_pennylane, available_adapters
print("adapters available here:", available_adapters())

qc = to_qiskit(c)
print("\nas a Qiskit circuit:")
print(qc.draw(output="text"))

adapters available here: ['qasm3', 'qiskit', 'pennylane']

as a Qiskit circuit:
     ┌─────────┐                
q_0: ┤ Ry(0.4) ├──────────■─────
     └─────────┘┌───┐     │     
q_1: ───────────┤ X ├─────■─────
        ┌───┐   └─┬─┘┌─────────┐
q_2: ───┤ H ├─────■──┤ Rz(0.7) ├
        └───┘        └─────────┘


### They agree

The point of an IR is that every emitter means the same thing. Let's check all three
executors produce the *same statevector*.

In [6]:
from qbm.circuits.adapters import executor

engines = ["builtin"] + [a for a in available_adapters() if a != "qasm3"]
reference = simulator.run(c)
print(f"{'engine':12s} max |psi - psi_builtin|")
print("-" * 36)
for name in engines:
    psi = executor(name)(c)
    print(f"{name:12s} {np.max(np.abs(psi - reference)):.2e}")

engine       max |psi - psi_builtin|
------------------------------------
builtin      0.00e+00
qiskit       0.00e+00
pennylane    0.00e+00


## Step 4 — swapping the engine for a whole QBM computation

`CircuitBackend` takes an `executor`, so you can route the *entire* QBM — Gibbs-state
preparation, Hadamard tests, the α-z information matrix — through whichever engine you
like, changing one argument.

In [7]:
ham2   = qbm.ParamHamiltonian(qbm.local_pauli_generators(2))
theta2 = np.random.default_rng(0).normal(scale=0.4, size=ham2.n_params)
dense2 = qbm.DenseBackend().thermal_state(ham2, theta2)
O2     = qbm.hamiltonians.tfim(2, g=1.2)
ref_metric = dense2.metric("wigner_yanase")

print(f"{'engine':12s} {'<O> error':>12s} {'alpha-z metric error':>22s}")
print("-" * 48)
for name in engines:
    st = CircuitBackend(seed=0, executor=executor(name)).thermal_state(ham2, theta2)
    est = st.metric(AlphaZ(0.5, 1.0))
    err_m = np.max(np.abs(est - ref_metric)) / np.max(np.abs(ref_metric))
    print(f"{name:12s} {abs(st.expect(O2) - dense2.expect(O2)):12.2e} {err_m:22.4f}")
print("\n(the metric error is Monte-Carlo sampling of the smearing time, identical")
print(" across engines because they compute the same thing)")

engine          <O> error   alpha-z metric error
------------------------------------------------
builtin          1.39e-16                 0.0059
qiskit           1.39e-16                 0.0059
pennylane        1.39e-16                 0.0059

(the metric error is Monte-Carlo sampling of the smearing time, identical
 across engines because they compute the same thing)


## Step 5 — what a hardware run would cost

Before sending anything to a device, ask what it needs. `resource_estimate()` reports
this up front rather than letting you discover it on a queue.

In [8]:
st = CircuitBackend(shots=10_000).thermal_state(ham, theta)
for k, v in st.resource_estimate(n_time_samples=100).items():
    print(f"   {k:24s} {v:,}")
print("\nthe metric costs ~J/2 times more circuits than a gradient -- which is why")
print("generative training is far more hardware-feasible than metric-based training.")

   n_qubits                 7
   prep_gates               1
   prep_depth               1
   n_parameters             8
   circuits_per_gradient    800
   circuits_per_metric      3,600
   shots_per_gradient       8,000,000
   shots_per_metric         36,000,000

the metric costs ~J/2 times more circuits than a gradient -- which is why
generative training is far more hardware-feasible than metric-based training.


## Writing your own emitter

An adapter is a function from our IR to something else. The whole contract is: walk
`circuit.gates` and translate. To support a new SDK (or a new hardware provider), copy
`qbm/circuits/adapters/qiskit_adapter.py` — about 70 lines — and change the gate names.

```python
def to_my_sdk(circuit):
    out = MySdkCircuit(circuit.n_qubits)
    for g in circuit.gates:
        if g.name == "h":     out.hadamard(g.qubits[0])
        elif g.name == "cx":  out.cnot(*g.qubits)
        elif g.name == "rz":  out.rz(g.params[0], g.qubits[0])
        ...
    return out
```

An `executor` is even simpler: any callable taking a `Circuit` and returning a
statevector can be passed to `CircuitBackend(executor=...)`.

## Recap

* The QBM algorithms use an internal IR and our own simulator — **no SDK is a
  dependency of the core**, verified here by blocking every SDK and running anyway.
* Emitters are thin leaves: OpenQASM 3 (no dependency), Qiskit, PennyLane.
* All engines produce identical results, so switching is a one-argument change.
* If Qiskit breaks tomorrow, the blast radius is one ~70-line file, and QASM export
  keeps working regardless.

That is the whole reason for the indirection: **your research code should outlive the
SDK release cycle.**